# Evaluate the Alignment Algorithm on Real Piano Performances

This notebook evaluates the edit-distance (ED) alignment algorithm in `compareMusic` on real piano performances from the [(n)ASAP dataset](https://github.com/CPJKU/asap-dataset) (Peter, 2023).

(n)ASAP is a dataset of aligned musical scores and performances built by extending the ASAP dataset with note-level annotations. The ASAP contains 236 distinct musical scores and 1067 performances of Western classical piano music from 15 different composers, the piece directory contains the XML and MIDI score, plus all of the performances of a specific piece, including: 
- `midi_score`: the MIDI score (what should be played) — used as **reference**
- `midi_performance`: what the pianist actually played — used as **response**
- `note_alignments.tsv`: official note-level alignment annotations — used as **ground truth**

bib citation:

@article{Peter-2023,
 title = {Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset},
 author = {Peter, Silvan David and Cancino-Chacón, Carlos Eduardo and Foscarin, Francesco and McLeod, Andrew Philip and Henkel, Florian and Karystinaios, Emmanouil and Widmer, Gerhard},
 doi = {10.5334/tismir.149},
 journal = {Transactions of the International Society for Music Information Retrieval {(TISMIR)}},
 year = {2023}
}

relevant paper: https://transactions.ismir.net/articles/10.5334/tismir.149#5-alignment-of-the-asap-dataset

## Download the (n)ASAP Dataset

In [1]:
import os

# Note: use CPJKU version (not fosfrancesco) because only CPJKU has
# the note_alignments TSV files are needed for ground truth comparison.
if not os.path.exists("asap-dataset"):
    os.system("git clone https://github.com/CPJKU/asap-dataset.git")
else:
    print("asap-dataset already exists, skipping download.")

ASAP_PATH = "asap-dataset"

asap-dataset already exists, skipping download.


## Load the Ground Truth (GT)

The `note_alignment.tsv` file in each performance contains the official note-level alignment annotations. Each row represents one note, with columns: `xml_id`, `midi_id`, `track`, `channel`, `pitch`, `onset`.
- If `midi_id == "deletion"`: the score note was not played → label `deletion`
- If `xml_id == "insertion"`: an extra note was played → label `insertion`
- Otherwise: a matched pair → label `paired`

For `match` and `insertion` rows, the TSV provides `onset` (onset time in seconds) and `pitch` (MIDI pitch) for the performance note.

In [2]:
import csv

def load_ground_truth(tsv_path):
    """
    Read a note_alignments TSV file from the CPJKU nASAP dataset.

    Args:
        tsv_path: str, path to the note_alignments.tsv file

    Returns:
        list of dicts, each with keys:
            label  -> 'paired', 'insertion', or 'deletion'
            onset  -> float (seconds) or None for deletions
            pitch  -> int (MIDI pitch number) or None for deletions
    """
    rows = []
    with open(tsv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        
        for row in reader:
            xml_id  = row["xml_id"].strip()
            midi_id = row["midi_id"].strip()

            if midi_id == "deletion":
                rows.append({"label": "deletion", "onset": None, "pitch": None})
            elif xml_id == "insertion":
                rows.append({
                    "label": "insertion",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
            else:
                rows.append({
                    "label": "paired",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
    return rows

## Load Score (reference)/Performance (response) pairs

Helper functions for format conversion: These functions convert a MIDI file into the `{pitch, start, duration}` format used by `compare_MIDI.py`.

In [3]:
import pretty_midi

def midi_file_to_notes(midi_path):
    """Parse a MIDI file into the format used by compareMusic."""
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []

    for instrument in midi_data.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            all_notes.append({
                "pitch": int(note.pitch),
                # Keep full precision. Rounding here can change tolerance-based
                # evaluation close to the matching boundary.
                "start": float(note.start),
                "duration": float(note.end - note.start),
            })

    all_notes.sort(key=lambda note: (note["start"], note["pitch"]))
    return all_notes

def build_sample(ref_path, response_path, composer, title, metadata_row):
    """Build one score/performance sample for compare_performance_ED."""
    score_notes = midi_file_to_notes(ref_path)
    performance_notes = midi_file_to_notes(response_path)

    if not score_notes or not performance_notes:
        return None

    return {
        "composer": composer,
        "title": title,
        "reference": {"notes": score_notes},
        "response": {"notes": performance_notes},
        "metadata_row": metadata_row,
    }

For the (n)ASAP dataset, the `metadata.csv` lists every score/performance pair in the dataset, check that both MIDI files exist on disk.

In [4]:
def load_samples(asap_path, composer):
    """
    Read the ASAP metadata CSV and return a list of sample dicts
    for a specific composer only.

    Args:
        asap_path: str, path to the cloned ASAP repo
        composer: str, e.g. "Bach"

    Returns:
        list of sample dicts (see build_sample)
    """
    metadata_path = os.path.join(asap_path, "metadata.csv")
    samples = []

    with open(metadata_path, "r", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            if row.get("composer", "").strip() == composer:
                ref_path = os.path.join(asap_path, row.get("midi_score", "").strip())
                response_path = os.path.join(asap_path, row.get("midi_performance", "").strip())

                if os.path.isfile(ref_path) and os.path.isfile(response_path):
                    sample = build_sample(
                        ref_path, response_path,
                        row.get("composer", "Unknown"),
                        row.get("title", "Unknown"),
                        dict(row),
                    )
                    if sample is not None:
                        samples.append(sample)

    return samples

samples = load_samples(ASAP_PATH, "Bach")

print("Loaded", len(samples), "score (reference) /performance (response) pairs.")

/Users/jz7125/compareMusic/.venv/lib/python3.13/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Loaded 169 score (reference) /performance (response) pairs.


# Run Alignment on Each Pair

Each score/performance pair is passed through `compare_performance_ED`. The current pipeline normalises the first onset to zero and groups nearby notes into events. Because `event_details` stores event indices, the same normalisation and grouping steps are reproduced here tso that the correct onset and pitch for each event during ground truth comparison can be looked up.

The original response onset offset is retained and added back only during comparison with the absolute performance times in the TSV file.

In [5]:
from evaluation_function.compare_MIDI import (
    compare_performance_ED,
    normalize_start_times,
    group_notes_into_events,
)

def run_alignment_on_samples(samples):
    """Run compare_performance_ED for every loaded sample."""
    results = []

    for sample in samples:
        result = compare_performance_ED(sample["response"], sample["reference"])

        response_notes_normalized = normalize_start_times(
            sample["response"]["notes"]
        )
        response_events_normalized = group_notes_into_events(
            response_notes_normalized
        )

        # Also rebuild the normalized reference events. convert_pipeline_output
        # needs these to look up how many notes are in a missing (deleted)
        # chord, since the pipeline's event_details does not carry that
        # information for missing/extra events.
        ref_notes_normalized = normalize_start_times(
            sample["reference"]["notes"]
        )
        ref_events_normalized = group_notes_into_events(
            ref_notes_normalized
        )

        response_onset_offset = float(sample["response"]["notes"][0]["start"])

        results.append({
            "composer": sample["composer"],
            "title": sample["title"],
            "stats": result.stats,
            "event_details": result.event_details,
            "is_correct": result.is_correct,
            "response_events_normalized": response_events_normalized,
            "ref_events_normalized": ref_events_normalized,
            "response_onset_offset": response_onset_offset,
        })

    return results

all_results = run_alignment_on_samples(samples)
print("Alignment complete for", len(all_results), "pieces.")

Alignment complete for 169 pieces.


## Compute Precision, Recall, and F1 

**Pipeline output vs ground truth**

he GT has three different labels, while our pipeline has four operations:

| Pipeline operation | GT label | Explanation |
|---|---|---|
| `match` | `paired` | Same pitch — GT still calls this `paired` |
| `replacement` | `paired` | Different pitch — GT still calls this `paired`, because the score note *was* played at approximately the right time |
| `missing` | `deletion` | Score note not found in response |
| `extra` | `insertion` | Response note has no score counterpart |

**Evaluation metrics design**

Evaluation is performed at **note level** after expanding chord events. Matches are one-to-one, so three notes at the same chord onset remain three separate observations.

| Label | Matching key | TP | FP | FN |
|---|---|---|---|---|
| `paired` (pipeline `match` / `replacement`) | onset-only matching within `ONSET_TOLERANCE`; pitch is intentionally ignored because this evaluates alignment rather than pitch correctness| Pipeline and ground truth both contain a paired label at matching performance onsets within the tolerance | Pipeline predicts a pairing at this onset that the ground truth does not recognise | Ground truth expects a pairing at this onset that the pipeline fails to find |
| `deletion` (pipeline `missing`) | count only as no onset or pitch available for deletions | Overlap between the pipeline deletion count and the ground-truth deletion count | The pipeline predicts more deletions than appear in the ground truth; the excess count is treated as FP | The ground truth contains more deletions than the pipeline predicts; the missing count is treated as FN |
| `insertion` (pipeline `extra`) | pitch must be equal and onset must be within `ONSET_TOLERANCE` | Pipeline and ground truth agree an extra, unexpected note was played at this pitch and onset | Pipeline flags a note as extra that the ground truth does not agree with | Ground truth marks a note as extra that the pipeline fails to identify |

- Precision: TP / (TP + FP), i.e. of all labels the pipeline predicted, how many were correct  
- Recall: TP / (TP + FN), i.e. of all GT labels, how many the pipeline found  
- F1: the harmonic mean of precision and recall 

In [6]:
def convert_pipeline_output(event_details, response_events, ref_events):
    """
    Convert event-level pipeline output into note-level labels.

    Note: ref_events is required here so that a missing (deleted) chord's
    true note count can be looked up. The pipeline's event_details does
    NOT carry pitch information for missing/extra events (correct_pitches
    and missing_pitches are always None in that case), so we must go back
    to the original reference events ourselves to know how many notes
    were actually deleted.
    """
    predictions = []

    for event in event_details:
        operation = event["operation_type"]
        event_type = event["event_type"]

        if operation == "missing":
            # A reference note/chord that the student did not play at all.
            if event_type == "note":
                number_of_notes = 1
            else:
                # reference_index is 1-based, so subtract 1 to index into ref_events.
                reference_index = event["reference_index"] - 1
                missing_ref_event = ref_events[reference_index]
                number_of_notes = len(missing_ref_event["notes"])

            for i in range(number_of_notes):
                predictions.append({
                    "onset": None,
                    "pitch": None,
                    "label": "deletion",
                })

        else:
            # response_index is 1-based in the pipeline output.
            response_index = event["response_index"] - 1
            if response_index < 0 or response_index >= len(response_events):
                raise IndexError("response_index is outside the response event list")

            response_event = response_events[response_index]
            onset = float(response_event["event_start"])

            if event_type == "note":
                pitch = int(response_event["notes"][0]["pitch"])
                if operation == "extra":
                    label = "insertion"
                else:
                    label = "paired"
                predictions.append({
                    "onset": onset,
                    "pitch": pitch,
                    "label": label,
                })

            elif operation == "extra":
                # Extra chord: every performed note in it is an insertion.
                for note in response_event["notes"]:
                    predictions.append({
                        "onset": onset,
                        "pitch": int(note["pitch"]),
                        "label": "insertion",
                    })

            else:
                # Matched or replaced chord. correct_pitches / missing_pitches /
                # extra_pitches already hold real MIDI pitches (see
                # compute_chord_accuracy in compare_MIDI.py), so we can use
                # them directly with no pitch-class fallback needed.
                for pitch in event.get("correct_pitches") or []:
                    predictions.append({
                        "onset": onset,
                        "pitch": int(pitch),
                        "label": "paired",
                    })

                for i in range(len(event.get("missing_pitches") or [])):
                    predictions.append({
                        "onset": None,
                        "pitch": None,
                        "label": "deletion",
                    })

                for pitch in event.get("extra_pitches") or []:
                    predictions.append({
                        "onset": onset,
                        "pitch": int(pitch),
                        "label": "insertion",
                    })

    return predictions

In [ ]:
ONSET_TOLERANCE = 0.05  # 50 milliseconds

def count_onset_matches(gt_onsets, predicted_onsets, tolerance):
    """Count one-to-one onset matches while preserving chord notes."""
    gt_onsets = sorted(gt_onsets)
    predicted_onsets = sorted(predicted_onsets)

    gt_index = 0
    predicted_index = 0
    matches = 0

    while gt_index < len(gt_onsets) and predicted_index < len(predicted_onsets):
        difference = predicted_onsets[predicted_index] - gt_onsets[gt_index]

        if abs(difference) <= tolerance:
            matches += 1
            gt_index += 1
            predicted_index += 1
        elif difference < -tolerance:
            predicted_index += 1
        else:
            gt_index += 1

    return matches

In [8]:
def compute_metrics(ground_truth, predictions, onset_offset=0.0):
    """Compute note-level precision, recall, and F1."""
    total_tp = 0
    total_fp = 0
    total_fn = 0

    # 1. Paired notes: compare onset only.
    gt_paired = [
        row["onset"]
        for row in ground_truth
        if row["label"] == "paired" and row["onset"] is not None
    ]
    predicted_paired = [
        row["onset"] + onset_offset
        for row in predictions
        if row["label"] == "paired" and row["onset"] is not None
    ]

    paired_tp = count_onset_matches(
        gt_paired, predicted_paired, ONSET_TOLERANCE
    )
    total_tp += paired_tp
    total_fp += len(predicted_paired) - paired_tp
    total_fn += len(gt_paired) - paired_tp

    # 2. Insertions: compare both pitch and onset.
    gt_insertions = {}
    predicted_insertions = {}

    for row in ground_truth:
        if (
            row["label"] == "insertion"
            and row["pitch"] is not None
            and row["onset"] is not None
        ):
            pitch = int(row["pitch"])
            if pitch not in gt_insertions:
                gt_insertions[pitch] = []
            gt_insertions[pitch].append(float(row["onset"]))

    for row in predictions:
        if (
            row["label"] == "insertion"
            and row["pitch"] is not None
            and row["onset"] is not None
        ):
            pitch = int(row["pitch"])
            if pitch not in predicted_insertions:
                predicted_insertions[pitch] = []
            predicted_insertions[pitch].append(
                float(row["onset"]) + onset_offset
            )

    for pitch in set(gt_insertions) | set(predicted_insertions):
        gt_onsets = gt_insertions.get(pitch, [])
        predicted_onsets = predicted_insertions.get(pitch, [])

        insertion_tp = count_onset_matches(
            gt_onsets,
            predicted_onsets,
            ONSET_TOLERANCE,
        )
        total_tp += insertion_tp
        total_fp += len(predicted_onsets) - insertion_tp
        total_fn += len(gt_onsets) - insertion_tp

    # 3. Deletions: count-based because the pipeline does not return xml_id.
    gt_deletions = sum(
        row["label"] == "deletion" for row in ground_truth
    )
    predicted_deletions = sum(
        row["label"] == "deletion" for row in predictions
    )

    deletion_tp = min(gt_deletions, predicted_deletions)
    total_tp += deletion_tp
    total_fp += predicted_deletions - deletion_tp
    total_fn += gt_deletions - deletion_tp

    precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp > 0 else 0.0
    )
    recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn > 0 else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
    }

## Evaluate all pieces

The evaluation is note-level. Notes inside a chord are kept as separate notes. Paired notes use a 50 ms onset tolerance, while insertions require both the same MIDI pitch and a matching onset.

In [ ]:
import pandas as pd

evaluation_rows = []

for sample, result in zip(samples, all_results):
    metadata_row = sample["metadata_row"]
    tsv_relative_path = (metadata_row.get("note_alignments") or "").strip()
    tsv_path = os.path.join(ASAP_PATH, tsv_relative_path)

    if not os.path.isfile(tsv_path):
        print("TSV not found:", tsv_path)
        continue

    ground_truth = load_ground_truth(tsv_path)
    predictions = convert_pipeline_output(
        result["event_details"],
        result["response_events_normalized"],
        result["ref_events_normalized"],
    )

    metrics = compute_metrics(
        ground_truth,
        predictions,
        onset_offset=result["response_onset_offset"],
    )

    evaluation_rows.append({
        "Piece": f'{result["composer"]} ({result["title"]})',
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
        "TP": metrics["tp"],
        "FP": metrics["fp"],
        "FN": metrics["fn"],
    })

df_eval = pd.DataFrame(evaluation_rows)

if df_eval.empty:
    print("No pieces were evaluated. Check the paths above.")
else:
    print(df_eval.head(10).round(4))

    print("Mean Precision:", round(df_eval["Precision"].mean(), 4))
    print("Mean Recall   :", round(df_eval["Recall"].mean(), 4))
    print("Mean F1       :", round(df_eval["F1"].mean(), 4))

                  Piece  Precision  Recall      F1    TP  FP  FN
0  Bach (Fugue_bwv_846)     0.9565  0.9739  0.9651   747  34  20
1  Bach (Fugue_bwv_848)     0.9945  0.9911  0.9928  1442   8  13
2  Bach (Fugue_bwv_848)     0.9809  0.9856  0.9833  1439  28  21
3  Bach (Fugue_bwv_848)     0.9788  0.9828  0.9808  1429  31  25
4  Bach (Fugue_bwv_848)     0.9863  0.9870  0.9866  1441  20  19
5  Bach (Fugue_bwv_848)     0.9945  0.9917  0.9931  1438   8  12
6  Bach (Fugue_bwv_848)     0.9897  0.9883  0.9890  1441  15  17
7  Bach (Fugue_bwv_848)     0.9689  0.9782  0.9735  1434  46  32
8  Bach (Fugue_bwv_848)     0.9945  0.9904  0.9924  1437   8  14
9  Bach (Fugue_bwv_848)     0.9910  0.9903  0.9907  1436  13  14
Mean Precision: 0.9644
Mean Recall   : 0.9743
Mean F1       : 0.9691


## Current limitation

Deletion matching is still count-based rather than identity-based. The ground-truth TSV identifies deleted score notes with `xml_id`, but the current pipeline output does not return that identifier, so we cannot yet confirm that a *specific* predicted deletion corresponds to a *specific* ground-truth deletion.

This version does fix an earlier bug where a missing (deleted) **chord** contributed 0 notes to the deletion count, because `event_details` leaves `correct_pitches`/`missing_pitches` as `None` for `missing`/`extra` events (this is true for both notes and chords, confirmed in `event_level_feedback()`). We now read the true note count for a missing chord directly from `ref_events` via `reference_index`, so the total deletion count is accurate again — only the note-to-note identity link is still missing.

A future improvement would be to keep the reference-note index or `xml_id` in each alignment result so that identity-based deletion matching becomes possible.